# HATS read path
Runs the `genesis/hats-reads` scenario of `ibench-storage hats` from a notebook and stores the records in `storage/results/`. Install with `uv sync --extra hats`, then register the environment as a Jupyter kernel (`uv run --extra hats python -m ipykernel install --user --name ibench`).

In [ ]:
import argparse
import os
import stat
from pathlib import Path

import pandas as pd

from ibench.cli import finish
from ibench.results import now
from ibench_storage import hats, profiles, scenario

In [ ]:
PROFILE = "s3"
CLIENT = "nersc-perlmutter"
SCENARIO = "genesis/hats-reads"
CACHE = "unknown"
REPEAT = 1
OVERRIDES = {"partitions": 16, "threads": [4]}
ENV_FILE = "~/.config/ibench/ro.env"

os.environ.setdefault("IBENCH_S3_ENDPOINT", "https://sdfdatas3.slac.stanford.edu")
os.environ.setdefault("IBENCH_S3_BUCKET", "rubin-benchmarks")

In [ ]:
if PROFILE == "s3":
    env_path = Path(ENV_FILE).expanduser()
    assert stat.S_IMODE(env_path.stat().st_mode) == 0o600, f"chmod 600 {env_path}"
    names = []
    for line in env_path.read_text().splitlines():
        line = line.strip().removeprefix("export ")
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            os.environ[k.strip()] = v.strip().strip("'\"")
            names.append(k.strip())
    print(names)

In [ ]:
prof = profiles.load(PROFILE)
spec = {**scenario.load(SCENARIO)["hats"], **OVERRIDES}
args = argparse.Namespace(client=CLIENT, site=None, cache=CACHE, tag=[], results_dir=None)
session = now()
for _ in range(REPEAT):
    runs = hats.run(prof, spec, SCENARIO)
    for r in runs:
        finish(r, args, "storage", session)

In [ ]:
pd.DataFrame([{"name": r.name, **{k: round(v, 3) for k, v in r.metrics.items() if k in
              ("gbps", "wall_s", "need_mb", "wire_mb", "over_read", "p50_ms")}} for r in runs])